In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
## 탐색적 데이터 분석
### 데이터 둘러보기
# pandas.read_csv()

data_path = '/kaggle/input/cat-in-the-dat/'

#train = pd.read_csv(data_path + 'train.csv')
train = pd.read_csv('/kaggle/input/cat-in-the-dat/train.csv')

#train.head()

train.shape

In [ ]:
train = pd.read_csv(data_path + 'train.csv', index_col='id')
test = pd.read_csv(data_path + 'test.csv', index_col='id')
submission = pd.read_csv(data_path + 'sample_submission.csv', index_col='id')

train.shape, test.shape

In [ ]:
pd.options.display.max_columns=24

train.head()

In [ ]:
train.head().T

#train.head().transpose()

In [ ]:
test.head()

In [ ]:
submission.head()

In [ ]:
### 피처 요약표
def resumetable(df):
    print(f'데이터셋 형상: {df.shape}')
    summary = pd.DataFrame(df.dtypes, columns=['데이터 타입'])
    summary = summary.reset_index()
    summary = summary.rename(columns={'index': '피처'})
    
    summary['결측값 개수'] = df.isnull().sum().values
    summary['고유값 개수'] = df.nunique().values
    summary['첫 번째 값'] = df.loc[0].values
    summary['두 번째 값'] = df.loc[1].values
    summary['세 번째 값'] = df.loc[2].values
    
    return summary

resumetable(train)

In [ ]:
### 피처 요약표 해석
### bin_0 ~ bin_4
### nom_0 ~ nom_9
### ord_0 ~ ord_5
### day, month, target

In [ ]:
### ord_0 ~ ord_2 의 고유값 출력
for i in range(3):
    feature = 'ord_' + str(i)
    print(f'{feature} 고유값: {train[feature].unique()}')

In [ ]:
### ord_3 ~ ord_5 의 고유값 출력
for i in range(3,6):
    feature = 'ord_' + str(i)
    print(f'{feature} 고유값: {train[feature].unique()}')

In [ ]:
### day, month, target 의 고유값 출력
print('day 고유값:', train['day'].unique())
print('month 고유값:', train['month'].unique())
print('target 고유값:', train['target'].unique())

In [ ]:
## 데이터 시각화

import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

In [ ]:
def write_percent(ax, total_size):
    for patch in ax.patches:
        height = patch.get_height()
        width = patch.get_width()
        left_coord = patch.get_x()
        percent = height/total_size*100
        
        ax.text(x=left_coord + width/2.0,
               y=height+total_size*0.001,
               s=f'{percent:1.1f}%',
               ha='center')

In [ ]:
### 타깃값 분포
### 수치형 데이터의 분포를 파악할 땐 주로 displot()을 사용
### 범주형 데이터의 분포를 파악할 땐 주로 countplot()을 사용

mpl.rc('font', size=15)
plt.figure(figsize=(5,4))

ax = sns.countplot(x='target', data=train)

ax.set_title('Target Distribution')

In [ ]:
mpl.rc('font', size=15)
plt.figure(figsize=(5,4))

ax = sns.countplot(x='target', data=train)

ax.set_title('Target Distribution')

write_percent(ax,len(train))

In [ ]:
### 이진 피처 분포

import matplotlib.gridspec as gridspec

# Step 1
mpl.rc('font', size=10)
grid = gridspec.GridSpec(3,2)
plt.figure(figsize=(6,10))
plt.subplots_adjust(wspace=0.6, hspace=0.4)

# Step 2
bin_features = ['bin_0','bin_1','bin_2','bin_3','bin_4']

#ax0 = plt.subplot(grid[0])
#ax0 = sns.countplot()

#ax1 = plt.subplot(grid[1])
#ax1 = sns.countplot()

for idx, feature in enumerate(bin_features):
    ax = plt.subplot(grid[idx])
    
    sns.countplot(x=feature,
                 data=train,
                 hue='target',
                 palette='pastel',
                 ax=ax)
    
# Step 3    
    ax.set_title(f'{feature} Distribution by Target')
    
    write_percent(ax, len(train))

In [ ]:
### 명목형 피처 (nom_0 ~ nom_4)
# Step 1
# 교차 분석표 생성 함수
# pandas의 crosstab() 함수
# 명목형 피처별 타깃값 비율을 구하기 위함

pd.crosstab(train['nom_0'], train['target'])

In [ ]:
# 정규화 후 비율을 백분율로 표현
crosstab = pd.crosstab(train['nom_0'], train['target'], normalize='index')*100
crosstab

In [ ]:
crosstab = crosstab.reset_index()
crosstab

In [ ]:
def get_crosstab(df, feature):
    crosstab = pd.crosstab(df[feature], df['target'], normalize='index')*100
    crosstab = crosstab.reset_index()
    return crosstab

In [ ]:
crosstab = get_crosstab(train, 'nom_0')
crosstab

In [ ]:
crosstab[0], crosstab[1]

In [ ]:
# Step 2
# 포인트플롯 생성 함수
def plot_pointplot(ax, feature, crosstab):
    ax2 = ax.twinx()  
    # x축은 공유하지만 y축은 공유하지 않는 새로운 축 ax2 생성
    # ax는 카운트플롯을 그리기 위한 축
    # ax2는 포인트플롯을 그리기 위한 축
    ax2 = sns.pointplot(x=feature, y=1, data=crosstab, 
                        order=crosstab[feature].values, 
                        color='black', 
                        legend=False)
    
    ax2.set_ylim(crosstab[1].min()-5, crosstab[1].max()*1.1)
    
    ax2.set_ylabel('Target 1 Ratio(%)')

In [ ]:
# Step 3
# 피처 분포도 및 피처별 타깃값의 비율 포인트플롯 생성
def plot_cat_dist_with_true_ratio(df, features, num_rows, num_cols, size=(15,20)):
    plt.figure(figsize=size)
    grid = gridspec.GridSpec(num_rows, num_cols)
    plt.subplots_adjust(wspace=0.45, hspace=0.3)
    
    for idx, feature in enumerate(features):
        ax = plt.subplot(grid[idx])
        crosstab = get_crosstab(df, feature)
        
        sns.countplot(x=feature, data=df,
                     order=crosstab[feature].values,
                     color='skyblue',
                     ax=ax)
        
        write_percent(ax, len(df))
        
        plot_pointplot(ax, feature, crosstab)
        
        ax.set_title(f'{feature} Distribution')

In [ ]:
nom_features = ['nom_0', 'nom_1', 'nom_2', 'nom_3', 'nom_4','nom_5']
plot_cat_dist_with_true_ratio(train, nom_features, num_rows=3, num_cols=2)

In [ ]:
### 순서형 피처
ord_features = ['ord_0', 'ord_1', 'ord_2', 'ord_3']
plot_cat_dist_with_true_ratio(train, ord_features, num_rows=2, num_cols=2, size=(15,12))

In [ ]:
# 순서형 피처에 순서지정
# CategoricalDtype()

from pandas.api.types import CategoricalDtype

ord_1_value = ['Novice', 'Contributor', 'Expert', 'Master', 'Grandmaster']
ord_2_value = ['Freezing', 'Cold', 'Warm', 'Hot', 'Boiling Hot', 'Lava Hot']

ord_1_dtype = CategoricalDtype(categories=ord_1_value, ordered=True)
ord_2_dtype = CategoricalDtype(categories=ord_2_value, ordered=True)

train['ord_1'] = train['ord_1'].astype(ord_1_dtype)
train['ord_2'] = train['ord_2'].astype(ord_2_dtype)

In [ ]:
plot_cat_dist_with_true_ratio(train, ord_features, num_rows=2, num_cols=2, size=(15,12))

In [ ]:
plot_cat_dist_with_true_ratio(train, ['ord_4','ord_5'], num_rows=2, num_cols=1, size=(15,12))

In [ ]:
# 날짜 피처
date_features = ['day', 'month']
plot_cat_dist_with_true_ratio(train, date_features, num_rows=2, num_cols=1, size=(10,10))

In [ ]:
## 베이스라인 모델

### 데이터 불러오기
import pandas as pd

data_path = '/kaggle/input/cat-in-the-dat/'

train = pd.read_csv(data_path+'train.csv', index_col='id')
test = pd.read_csv(data_path+'test.csv', index_col='id')
submission = pd.read_csv(data_path+'sample_submission.csv', index_col='id')

In [ ]:
### 피처 엔지니어링
# 데이터 합치기
all_data = pd.concat([train, test])

all_data.head()

In [ ]:
all_data = all_data.drop('target', axis=1)

all_data.head()

In [ ]:
all_data.shape

In [ ]:
### 원-핫 인코딩
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder()
all_data_encoded = encoder.fit_transform(all_data)

all_data_encoded

In [ ]:
print(all_data_encoded)

In [ ]:
### 데이터 나누기

num_train = len(train)

X_train = all_data_encoded[:num_train]
X_test = all_data_encoded[num_train:]

y = train['target']


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(X_train, y,
                                                     test_size=0.1,
                                                     stratify=y,
                                                     random_state=10)

In [ ]:
### 모델 훈련

from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(max_iter=1000, random_state=42)

logistic_model.fit(X_train, y_train)

In [ ]:
### 모델 성능 검증
# predict(): 타깃값 자체 0 혹은 1을 예측
# predict_proba() : 타깃값의 확률을 예측

logistic_model.predict_proba(X_valid)

In [ ]:
logistic_model.predict(X_valid)

In [ ]:
y_valid_preds = logistic_model.predict_proba(X_valid)[:,1]

In [ ]:
# ROC AUC 평가함수
from sklearn.metrics import roc_auc_score

roc_auc = roc_auc_score(y_valid, y_valid_preds)

print(f'검증 데이터 ROC AUC : {roc_auc: .4f}')

In [ ]:
# 타깃값이 1일 확률 예측
#y_preds = logistic_model.predict_proba(X_test)[:,1]

In [ ]:
### 제출

#submission['target'] = y_preds
#submission.to_csv('submission.csv')

In [ ]:
## 성능 개선 1
# 1. 피처 맞춤 인코딩
# 이진 피처와 순서형 피처 ord_1, ord_2는 수작업으로 인코딩 
# 순서형 피처 ord_3, ord_4, ord_5는 ordinal 인코딩
# 명목형 피처와 날짜 피처는 원-핫 인코딩

# 2. 피처 스케일링 : 피처 간 값의 범위를 일치시키는 작업
# 순서형 피처에만 적용

# 3. 하이퍼파라미터 최적화
# 그리드서치

In [ ]:
### 데이터 불러오기
import pandas as pd

data_path = '/kaggle/input/cat-in-the-dat/'

train = pd.read_csv(data_path+'train.csv', index_col='id')
test = pd.read_csv(data_path+'test.csv', index_col='id')
submission = pd.read_csv(data_path+'sample_submission.csv', index_col='id')

In [ ]:
### 피처 엔지니어링
# 1. 피처 맞춤 인코딩

# 데이터 합치기
all_data = pd.concat([train, test])

all_data = all_data.drop('target', axis=1)

# -------------------------------------------
# 이진 피처 인코딩
# bin_0, bin_1, bin_2 피처는 0과 1
# bin_3은 T와 F, bin_4는 Y와 N
all_data['bin_3'] = all_data['bin_3'].map({'F':0, 'T':1})
all_data['bin_4'] = all_data['bin_4'].map({'N':0, 'Y':1})

# -------------------------------------------
# 순서형 피처 인코딩
# ord_0 피처는 이미 숫자
# ord_1과 ord_2 인코딩
ord1dict = {'Novice':0, 'Contributor':1, 'Expert':2, 'Master':3, 'Grandmaster':4}
ord2dict = {'Freezing':0, 'Cold':1, 'Warm':2, 'Hot':3, 'Boiling Hot':4, 'Lava Hot':5}

all_data['ord_1'] = all_data['ord_1'].map(ord1dict)
all_data['ord_2'] = all_data['ord_2'].map(ord2dict)


from sklearn.preprocessing import OrdinalEncoder

ord_345 = ['ord_3','ord_4','ord_5']

ord_encoder = OrdinalEncoder()

all_data[ord_345] = ord_encoder.fit_transform(all_data[ord_345])

# -------------------------------------------
# 명목형 피처 인코딩
# 순서를 무시해도 무관하므로 원-핫 인코딩

nom_features = ['nom_'+str(i) for i in range(10)]

from sklearn.preprocessing import OneHotEncoder

onehot_encoder = OneHotEncoder()

encoded_nom_matrix = onehot_encoder.fit_transform(all_data[nom_features])

all_data = all_data.drop(nom_features, axis=1)

# -------------------------------------------
# 날짜 피처 인코딩
date_features = ['day','month']

encoded_date_matrix = onehot_encoder.fit_transform(all_data[date_features])

all_data = all_data.drop(date_features, axis=1)

In [ ]:
# 2. 피처 스케일링
# 서로 다른 피처들의 값의 범위가 일치하도록 조정하는 작업
# 수치형 피처들의 유효값 범위가 서로 다르면 훈련이 제대로 안 될 수도 있기 때문에 필요
# 순서형 피처도 0~1 사이가 되도록 스케일링

# 순서형 피처 스케일링
from sklearn.preprocessing import MinMaxScaler

ord_features = ['ord_'+str(i) for i in range(6)]

all_data[ord_features] = MinMaxScaler().fit_transform(all_data[ord_features])

In [ ]:
### 인코딩 및 스케일링된 피처 합치기
from scipy import sparse

all_data_sprs = sparse.hstack([sparse.csr_matrix(all_data),
                              encoded_nom_matrix,
                              encoded_date_matrix],
                             format='csr')

In [ ]:
### 데이터 나누기

num_train = len(train)

X_train = all_data_sprs[:num_train]
X_test = all_data_sprs[num_train:]

y = train['target']

In [ ]:
all_data_sprs

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(X_train, y,
                                                     test_size=0.1,
                                                     stratify=y,
                                                     random_state=10)

In [ ]:
# 3. 하이퍼파라미터 최적화
#%%time

from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression()

lr_params = {'C':[0.1,0.125,0.2], 'max_iter':[800,900,1000],'solver':['liblinear'], 'random_state': [42]}

gridsearch_logistic_model = GridSearchCV(estimator=logistic_model,
                                        param_grid=lr_params,
                                        scoring='roc_auc',
                                        cv=5)

gridsearch_logistic_model.fit(X_train, y_train)

print('최적 하리퍼파라미터: ', gridsearch_logistic_model.best_params_)

In [ ]:
### 모델 성능 검증
from sklearn.metrics import roc_auc_score

y_valid_preds = gridsearch_logistic_model.predict_proba(X_valid)[:,1]

roc_auc = roc_auc_score(y_valid, y_valid_preds)

print(f'검증 데이터 ROC AUC: {roc_auc:.4f}')

In [ ]:
### 결과 제출
#y_preds = gridsearch_logistic_model.best_estimator_.predict_proba(X_test)[:,1]

#submission['target'] = y_preds
#submission.to_csv('submission.csv')

In [ ]:
## 성능 개선 2
### 데이터 불러오기
import pandas as pd

data_path = '/kaggle/input/cat-in-the-dat/'

train = pd.read_csv(data_path+'train.csv', index_col='id')
test = pd.read_csv(data_path+'test.csv', index_col='id')
submission = pd.read_csv(data_path+'sample_submission.csv', index_col='id')

### 피처 엔지니어링
# 1. 피처 맞춤 인코딩

# 데이터 합치기
all_data = pd.concat([train, test])

all_data = all_data.drop('target', axis=1)

# -------------------------------------------
# 이진 피처 인코딩
# bin_0, bin_1, bin_2 피처는 0과 1
# bin_3은 T와 F, bin_4는 Y와 N
all_data['bin_3'] = all_data['bin_3'].map({'F':0, 'T':1})
all_data['bin_4'] = all_data['bin_4'].map({'N':0, 'Y':1})

# -------------------------------------------
# 순서형 피처 인코딩
# ord_0 피처는 이미 숫자
# ord_1과 ord_2 인코딩
ord1dict = {'Novice':0, 'Contributor':1, 'Expert':2, 'Master':3, 'Grandmaster':4}
ord2dict = {'Freezing':0, 'Cold':1, 'Warm':2, 'Hot':3, 'Boiling Hot':4, 'Lava Hot':5}

all_data['ord_1'] = all_data['ord_1'].map(ord1dict)
all_data['ord_2'] = all_data['ord_2'].map(ord2dict)


from sklearn.preprocessing import OrdinalEncoder

ord_345 = ['ord_3','ord_4','ord_5']

ord_encoder = OrdinalEncoder()

all_data[ord_345] = ord_encoder.fit_transform(all_data[ord_345])

# -------------------------------------------
# 명목형 피처 인코딩
# 순서를 무시해도 무관하므로 원-핫 인코딩

nom_features = ['nom_'+str(i) for i in range(10)]

from sklearn.preprocessing import OneHotEncoder

onehot_encoder = OneHotEncoder()

encoded_nom_matrix = onehot_encoder.fit_transform(all_data[nom_features])

all_data = all_data.drop(nom_features, axis=1)

# -------------------------------------------
# 날짜 피처 인코딩
date_features = ['day','month']

encoded_date_matrix = onehot_encoder.fit_transform(all_data[date_features])

all_data = all_data.drop(date_features, axis=1)

# 2. 피처 스케일링
# 서로 다른 피처들의 값의 범위가 일치하도록 조정하는 작업
# 수치형 피처들의 유효값 범위가 서로 다르면 훈련이 제대로 안 될 수도 있기 때문에 필요
# 순서형 피처도 0~1 사이가 되도록 스케일링

# 순서형 피처 스케일링
from sklearn.preprocessing import MinMaxScaler

ord_features = ['ord_'+str(i) for i in range(6)]

all_data[ord_features] = MinMaxScaler().fit_transform(all_data[ord_features])

### 인코딩 및 스케일링된 피처 합치기
from scipy import sparse

all_data_sprs = sparse.hstack([sparse.csr_matrix(all_data),
                              encoded_nom_matrix,
                              encoded_date_matrix],
                             format='csr')

### 데이터 나누기

num_train = len(train)

X_train = all_data_sprs[:num_train]
X_test = all_data_sprs[num_train:]

y_train = train['target']

# 3. 하이퍼파라미터 최적화
#%%time

from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression()

lr_params = {'C':[0.1,0.125,0.2], 'max_iter':[800,900,1000],'solver':['liblinear'], 'random_state': [42]}

gridsearch_logistic_model = GridSearchCV(estimator=logistic_model,
                                        param_grid=lr_params,
                                        scoring='roc_auc',
                                        cv=5)

gridsearch_logistic_model.fit(X_train, y_train)

print('최적 하리퍼파라미터: ', gridsearch_logistic_model.best_params_)


### 결과 제출
y_preds = gridsearch_logistic_model.best_estimator_.predict_proba(X_test)[:,1]

submission['target'] = y_preds
submission.to_csv('submission.csv')